<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/matrix_transpose_kernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from numba import cuda
import cupy as cp
import numpy as np


@cuda.jit
def transpose(input_mat,output_mat):

  n,m = output_mat.shape
  out_row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
  out_col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x

  if out_col < m and out_row < n:
    output_mat[out_row, out_col] = input_mat[out_col, out_row]


def main():

  m = 10000
  n = 2000

  device = cp.cuda.Device()

  input_mat = cp.random.uniform(-1,1, (m,n), dtype = cp.float32)
  output_mat = cp.zeros((n,m), dtype = cp.float32)


  #launch configuration
  threads_per_block = (32, 32)
  blocks_per_grid = (
      (m + threads_per_block[0] - 1) // threads_per_block[0],
      (n + threads_per_block[1] - 1) // threads_per_block[1],
  )

  transpose[blocks_per_grid, threads_per_block](input_mat, output_mat)

  cuda.synchronize()

  input_mat_np = cp.asnumpy(input_mat)
  output_mat_np = cp.asnumpy(output_mat)
  np.testing.assert_allclose(input_mat_np.T, output_mat_np, rtol=1e-2,atol=1e-2)

  print(input_mat_np)
  print(output_mat_np)

main()

[[-0.7897478  -0.93127954  0.74275553 ...  0.16031969 -0.30996013
   0.33460093]
 [ 0.4198861  -0.84464085  0.34663224 ...  0.22155333  0.6294918
  -0.6132361 ]
 [-0.5006536   0.15764534 -0.78657776 ... -0.24585068 -0.09182394
   0.38558805]
 ...
 [-0.05054718  0.3156947  -0.9814776  ... -0.8803465  -0.5490946
   0.484218  ]
 [ 0.43781686  0.44423115  0.67052746 ...  0.22022223  0.45591664
  -0.13240904]
 [ 0.92973363 -0.21111703 -0.06044012 ... -0.7012279   0.05336905
  -0.19434011]]
[[-0.7897478   0.4198861  -0.5006536  ... -0.05054718  0.43781686
   0.92973363]
 [-0.93127954 -0.84464085  0.15764534 ...  0.3156947   0.44423115
  -0.21111703]
 [ 0.74275553  0.34663224 -0.78657776 ... -0.9814776   0.67052746
  -0.06044012]
 ...
 [ 0.16031969  0.22155333 -0.24585068 ... -0.8803465   0.22022223
  -0.7012279 ]
 [-0.30996013  0.6294918  -0.09182394 ... -0.5490946   0.45591664
   0.05336905]
 [ 0.33460093 -0.6132361   0.38558805 ...  0.484218   -0.13240904
  -0.19434011]]


In [40]:
%%writefile transpose-naive.py
from numba import cuda
import cupy as cp
import numpy as np


@cuda.jit
def transpose(input_mat,output_mat):

  n,m = output_mat.shape
  out_row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
  out_col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x

  if out_col < m and out_row < n:
    output_mat[out_row, out_col] = input_mat[out_col, out_row]


def main():

  m = 10000
  n = 2000

  device = cp.cuda.Device()

  input_mat = cp.random.uniform(-1,1, (m,n), dtype = cp.float32)
  output_mat = cp.zeros((n,m), dtype = cp.float32)


  #launch configuration
  threads_per_block = (32, 32)
  blocks_per_grid = (
      (m + threads_per_block[0] - 1) // threads_per_block[0],
      (n + threads_per_block[1] - 1) // threads_per_block[1],
  )

  transpose[blocks_per_grid, threads_per_block](input_mat, output_mat)

  cuda.synchronize()

  input_mat_np = cp.asnumpy(input_mat)
  output_mat_np = cp.asnumpy(output_mat)
  np.testing.assert_allclose(input_mat_np.T, output_mat_np, rtol=1e-2,atol=1e-2)

  print(input_mat_np)
  print(output_mat_np)

main()

Writing transpose-naive.py


In [41]:
!ncu --set full -f -o transpose-naive.ncu-rep python3 transpose-naive.py

==PROF== Connected to process 29390 (/usr/bin/python3.13)
==PROF== Profiling "generate_seed_pseudo" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "gen_sequenced" - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_random_x_mod_1" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_scale" - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "transpose[abi:v1,cw51cXTLSUwv..." - 4: 0%....50%....100% - 33 passes
[[ 0.01374328  0.3322518  -0.57459867 ...  0.4468907  -0.6348117
   0.36449015]
 [ 0.5316776   0.42408955  0.5726681  ...  0.6274719  -0.09567839
   0.26607692]
 [-0.70869875 -0.2552557   0.9737395  ...  0.60541224 -0.40587652
   0.3715061 ]
 ...
 [ 0.39348638  0.26228642  0.5748682  ...  0.55905795 -0.3080238
  -0.74061966]
 [-0.6859597   0.5732626  -0.978134   ... -0.88034666 -0.19575876
   0.37712026]
 [ 0.8940716   0.45294678  0.6054733  ...  0.11583555  0.07098413
  -0.32109475]]
[[ 0.01374328  0.5316776  -0.70869875 ...  0.39348638 -0.6859597
   0.894